<a href="https://colab.research.google.com/github/JE7500/AI-Based-Rockfall-Prediction-Alert-System-for-open-pit-mines/blob/main/rockfall_inference_colab_fixed_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Rockfall Model Inference — Fixed Google Colab Notebook (v2)

This notebook runs inference using `rockfall_models_small.zip` and your input CSV.
It includes robust checks for the `models` directory and artifact filenames, and avoids the previous string-quote issue.

**Upload when prompted:** the model ZIP (e.g., `rockfall_models_small.zip`) and your input CSV (e.g., `open_pit_mine_rockfall_dataset_sih.csv`).
Run cells top-to-bottom.


In [1]:
# Install required packages
!pip install --upgrade pip
!pip install scikit-learn joblib pandas matplotlib
print('Packages installed.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 27.4 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
Packages installed.


In [6]:
# Upload the model zip and your CSV file when prompted.
from google.colab import files
import os

print("Please upload the model zip (rockfall_models_small.zip) and your CSV file now.")
uploaded = files.upload()  # upload dialog appears - select your files

print("\nUploaded files:")
for fn in uploaded.keys():
    print('-', fn)

# List current files
print("\nCurrent working directory contents:")
print(os.listdir('.'))

Please upload the model zip (rockfall_models_small.zip) and your CSV file now.


Saving open_pit_mine_rockfall_dataset_sih.csv to open_pit_mine_rockfall_dataset_sih.csv

Uploaded files:
- open_pit_mine_rockfall_dataset_sih.csv

Current working directory contents:
['.config', 'rockfall_models_small.zip', 'models', 'open_pit_mine_rockfall_dataset_sih.csv', 'sample_data']


In [3]:
# Prepare ./models directory: extract zip(s) if necessary and ensure expected files exist.
import os, zipfile, shutil

os.makedirs('models', exist_ok=True)
print("Ensured ./models directory exists. Current contents:", os.listdir('models'))

# If models folder is empty, try extract the first zip in cwd
if not os.listdir('models'):
    zip_files = [f for f in os.listdir('.') if f.lower().endswith('.zip')]
    if not zip_files:
        print("\nNo zip file found in the working directory. If you haven't uploaded, re-run the upload cell and upload the zip.")
    else:
        zip_path = zip_files[0]
        print("\nExtracting zip:", zip_path)
        with zipfile.ZipFile(zip_path, 'r') as zf:
            zf.extractall('models')
        print("Extraction complete. ./models now contains:", os.listdir('models'))

# Helper: try to rename or copy artifacts so that the notebook can find them by standard names.
expected_map = {
    'rf_classifier.joblib': ['rf_classifier.joblib', 'rf_classifier_small.joblib', 'classifier.joblib'],
    'rf_regressor.joblib': ['rf_regressor.joblib', 'rf_regressor_small.joblib', 'regressor.joblib'],
    'standard_scaler.joblib': ['standard_scaler.joblib', 'standard_scaler_small.joblib', 'scaler.joblib'],
    'feature_columns.json': ['feature_columns.json', 'feature_columns_small.json'],
    'numeric_columns.json': ['numeric_columns.json', 'numeric_columns_small.json']
}

found = os.listdir('models')
print("\nArtifacts currently in models/:", found)

for target, candidates in expected_map.items():
    # if target already exists, continue
    target_path = os.path.join('models', target)
    if os.path.exists(target_path):
        continue
    # search for candidate files (exact or partial match)
    for fn in found:
        for cand in candidates:
            if fn == cand or cand.replace('.joblib','').replace('.json','') in fn:
                src = os.path.join('models', fn)
                # copy to standardized name if necessary
                try:
                    shutil.copy2(src, target_path)
                    print(f'Copied {fn} -> {target}')
                except Exception as e:
                    print('Error copying', fn, '->', target, e)
                break

# Final check
print("\nFinal models/ contents:", os.listdir('models'))

# If still missing artifacts, show helpful message
missing = [t for t in expected_map.keys() if not os.path.exists(os.path.join('models', t))]
if missing:
    print("\nWARNING: The following expected artifacts are missing from ./models:", missing)
    print("Please recheck the uploaded zip or upload the correct files.")
else:
    print("\nAll expected artifacts are present. You can proceed to run the inference cell.")

Ensured ./models directory exists. Current contents: []

Extracting zip: rockfall_models_small.zip
Extraction complete. ./models now contains: ['standard_scaler_small.joblib', 'rf_regressor_small.joblib', 'numeric_columns_small.json', 'rf_classifier_small.joblib', 'feature_columns_small.json']

Artifacts currently in models/: ['standard_scaler_small.joblib', 'rf_regressor_small.joblib', 'numeric_columns_small.json', 'rf_classifier_small.joblib', 'feature_columns_small.json']
Copied rf_classifier_small.joblib -> rf_classifier.joblib
Copied rf_regressor_small.joblib -> rf_regressor.joblib
Copied standard_scaler_small.joblib -> standard_scaler.joblib
Copied feature_columns_small.json -> feature_columns.json
Copied numeric_columns_small.json -> numeric_columns.json

Final models/ contents: ['feature_columns.json', 'standard_scaler_small.joblib', 'rf_regressor_small.joblib', 'rf_regressor.joblib', 'numeric_columns_small.json', 'rf_classifier_small.joblib', 'feature_columns_small.json', 'rf_

In [7]:
import os, json, joblib, pandas as pd, numpy as np
from google.colab import files

models_dir = 'models'

# Verify models_dir exists and has expected artifacts
if not os.path.isdir(models_dir):
    raise FileNotFoundError("Directory './models' not found. Run the previous cells to upload and extract the models zip.")

expected_files = ['rf_classifier.joblib', 'rf_regressor.joblib', 'standard_scaler.joblib', 'feature_columns.json', 'numeric_columns.json']
missing = [f for f in expected_files if not os.path.exists(os.path.join(models_dir, f))]
if missing:
    raise FileNotFoundError(f"The following required files are missing in './models': {missing}. Ensure you've uploaded the correct zip and re-run the extraction cell.")

# Load artifacts
print('Loading artifacts from', models_dir)
clf = joblib.load(os.path.join(models_dir, 'rf_classifier.joblib'))
reg = joblib.load(os.path.join(models_dir, 'rf_regressor.joblib'))
scaler = joblib.load(os.path.join(models_dir, 'standard_scaler.joblib'))

with open(os.path.join(models_dir, 'feature_columns.json'), 'r') as f:
    feature_cols = json.load(f)
with open(os.path.join(models_dir, 'numeric_columns.json'), 'r') as f:
    numeric_cols = json.load(f)

print('Artifacts loaded. Model expects', len(feature_cols), 'features.')

# Find input CSV file in cwd (ignore known sample file)
csv_candidates = [f for f in os.listdir('.') if f.lower().endswith('.csv') and 'predictions' not in f.lower()]
if not csv_candidates:
    raise FileNotFoundError("No CSV input found in working directory. Upload your input CSV (e.g., open_pit_mine_rockfall_dataset_sih.csv) using the upload cell and re-run this cell.")
input_csv = csv_candidates[0]
print('Using input CSV:', input_csv)

# Read CSV, parse timestamp if present
df = pd.read_csv(input_csv)
if 'Timestamp' in df.columns:
    df['Timestamp'] = pd.to_datetime(df['Timestamp'], errors='coerce')

# Preprocess (timestamp features, one-hot rock type)
def preprocess(df):
    df = df.copy()
    if 'Timestamp' in df.columns:
        df['hour'] = df['Timestamp'].dt.hour.fillna(0).astype(int)
        df['day'] = df['Timestamp'].dt.day.fillna(0).astype(int)
        df['month'] = df['Timestamp'].dt.month.fillna(0).astype(int)
        df['year'] = df['Timestamp'].dt.year.fillna(0).astype(int)
        df['dayofweek'] = df['Timestamp'].dt.dayofweek.fillna(0).astype(int)
        df['is_weekend'] = df['dayofweek'].isin([5,6]).astype(int)
        df = df.drop(columns=['Timestamp'])
    else:
        df['hour'] = 0
        df['day'] = 0
        df['month'] = 0
        df['year'] = 0
        df['dayofweek'] = 0
        df['is_weekend'] = 0
    if 'Rock_Type_Category' in df.columns:
        df = pd.get_dummies(df, columns=['Rock_Type_Category'], prefix='rock')
    return df

df_proc = preprocess(df)

# Align features: ensure all feature_cols exist; if extras present, drop them
for col in feature_cols:
    if col not in df_proc.columns:
        df_proc[col] = 0.0
X = df_proc[feature_cols].copy()

# Numeric columns fallback
numeric_present = [c for c in numeric_cols if c in X.columns]
if len(numeric_present) == 0:
    numeric_present = X.select_dtypes(include=[np.number]).columns.tolist()
print('Numeric columns used for scaling:', numeric_present)

# Scale numeric features
X_scaled = X.copy()
X_scaled[numeric_present] = scaler.transform(X[numeric_present])

# Predictions
pred_alert = clf.predict(X_scaled)
pred_alert_proba = clf.predict_proba(X_scaled)[:,1] if hasattr(clf, 'predict_proba') else None
pred_risk = reg.predict(X_scaled)

# Attach predictions and save
out = df.copy().reset_index(drop=True)
out['pred_alert_status'] = pred_alert
if pred_alert_proba is not None:
    out['pred_alert_proba'] = pred_alert_proba
out['pred_risk_score'] = pred_risk

out_file = 'predictions_from_colab_fixed_v2.csv'
out.to_csv(out_file, index=False)
print('Saved predictions to', out_file)
files.download(out_file)

Loading artifacts from models
Artifacts loaded. Model expects 6 features.
Using input CSV: open_pit_mine_rockfall_dataset_sih.csv
Numeric columns used for scaling: ['Slope_Angle', 'Rock_Strength', 'Moisture']
Saved predictions to predictions_from_colab_fixed_v2.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np

print("\n--- CLASSIFIER EVALUATION ---")
# Only if ground truth alert status exists in CSV
if 'Alert_status' in df.columns:
    y_true = df['Alert_status']
    y_pred = pred_alert

    acc = accuracy_score(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred)
    report = classification_report(y_true, y_pred)

    print("Accuracy:", acc)
    print("Confusion Matrix:\n", cm)
    print("Classification Report:\n", report)
else:
    print("No 'Alert_status' column found in the input CSV → cannot compute classifier accuracy.")

print("\n--- REGRESSOR EVALUATION ---")
# Only if ground truth risk score exists in CSV
if 'Risk_Score' in df.columns:   # replace with the actual target column name used in training
    y_true_reg = df['Risk_Score']
    y_pred_reg = pred_risk

    r2 = r2_score(y_true_reg, y_pred_reg)
    mae = mean_absolute_error(y_true_reg, y_pred_reg)
    rmse = np.sqrt(mean_squared_error(y_true_reg, y_pred_reg))

    print("R² Score:", r2)
    print("MAE:", mae)
    print("RMSE:", rmse)
else:
    print("No 'Risk_Score' column found in the input CSV → cannot compute regressor accuracy.")


--- CLASSIFIER EVALUATION ---
No 'Alert_status' column found in the input CSV → cannot compute classifier accuracy.

--- REGRESSOR EVALUATION ---
No 'Risk_Score' column found in the input CSV → cannot compute regressor accuracy.


---
If you still see an error, copy the **full** traceback here and I'll debug it immediately.
